In [1]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from pprint import pformat
formatter = get_ipython().display_formatter.formatters['text/plain']
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))

<function IPython.lib.pretty._seq_pprinter_factory.<locals>.inner(obj, p, cycle)>

In [2]:
import random
from dataclasses import asdict
from datasets import load_from_disk
from tqdm.auto import tqdm

from src.editors import EditRequest, load_editor
from src.evaluate import EvalPair, EvalSuite, evaluate, save_results

In [3]:
# --- Configuration ---
method = ["ft", "rome", "memit", "grace", "wise", "alpha_edit"][5]
model = ["gpt2-xl", "gpt-j-6b", "qwen2.5-7b", "llama-3-8b"][0]

steps = 1          # number of sequential edit steps
non_seq_per_step = 0 # non-sequential edits per step

In [4]:
editor = load_editor(f"../src/config/{method}/{model}.yaml")
cfact = load_from_disk("../data/datasets/cfact")
mquake = load_from_disk("../data/datasets/mquake")

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

In [5]:
# --- Helper: build eval data from a cfact sample ---
def cfact_eval_data(sample):
    request = EditRequest(
        subject=sample["subject"],
        prompt=sample["prompt"],
        old_ans=sample["old_ans"],
        new_ans=sample["new_ans"],
    )

    reliability_pairs = [EvalPair(prompt=sample["prompt"], expected_ans=sample["new_ans"])]
    for rephrase in sample["rephrases"]:
        reliability_pairs.append(EvalPair(prompt=rephrase, expected_ans=sample["new_ans"]))

    locality_pairs = [EvalPair(prompt=loc["prompt"], expected_ans=loc["ans"])
                      for loc in sample["locality"]]

    suite = EvalSuite(
        reliability=reliability_pairs,
        locality={"locality": locality_pairs} if locality_pairs else None,
    )
    return request, suite


# --- Helper: build eval data from a mquake sample ---
def mquake_eval_data(sample):
    request = EditRequest(
        subject=sample["subject"],
        prompt=sample["prompt"],
        old_ans=sample["old_ans"],
        new_ans=sample["new_ans"],
    )
    port_pairs = []
    for port in sample["portability"]:
        port_pairs.append(EvalPair(prompt=port["prompt"], expected_ans=port["new_ans"]))
        for rephrase in port["rephrases"]:
            port_pairs.append(EvalPair(prompt=rephrase, expected_ans=port["new_ans"]))

    suite = EvalSuite(
        portability={"portability": port_pairs} if port_pairs else None,
    )
    return request, suite


# --- Helper: select samples for a dataset ---
def select_samples(dataset, n_steps, n_nonseq_per_step):
    total = n_steps * (1 + n_nonseq_per_step)
    indices = random.sample(range(len(dataset)), total)
    seq_indices = indices[:n_steps]
    nonseq_indices = indices[n_steps:]
    return seq_indices, nonseq_indices

# Phase 1: Counterfact (Reliability + Locality)

Each step:
1. 1 sequential edit → evaluate
2. (From step 2+) Re-evaluate all previous sequential edits (stability)

Step score = total matched / total records across all evaluations in the step, including stability (reliability & locality kept separate).

In [6]:
# --- Sample selection for cfact ---
cfact_seq_indices, cfact_nonseq_indices = select_samples(cfact, steps, non_seq_per_step)
print(f"cfact: {steps} sequential samples, {len(cfact_nonseq_indices)} non-sequential samples")

cfact: 1 sequential samples, 0 non-sequential samples


In [7]:
editor.reset()

cfact_results = []
saved_seq_suites: list[EvalSuite] = []

for step in tqdm(range(steps), desc="cfact"):
    step_num = step + 1
    step_rel_matched, step_rel_total = 0, 0
    step_loc_matched, step_loc_total = 0, 0
    step_edits = []

    # --- Non-sequential edits ---
    for j in range(non_seq_per_step):
        ns_idx = cfact_nonseq_indices[step * non_seq_per_step + j]
        sample = cfact[ns_idx]
        request, suite = cfact_eval_data(sample)

        editor.edit(request, is_sequential=False)
        result = evaluate(editor.model, editor.tokenizer, suite)

        if result.reliability:
            step_rel_matched += sum(r.match for r in result.reliability.details)
            step_rel_total += len(result.reliability.details)
        if result.locality:
            for mr in result.locality.values():
                step_loc_matched += sum(r.match for r in mr.details)
                step_loc_total += len(mr.details)

        step_edits.append({
            "type": "non_sequential",
            "dataset_index": ns_idx,
            "edit_request": asdict(request),
            **result.to_dict(),
        })

    # --- Sequential edit ---
    seq_idx = cfact_seq_indices[step]
    sample = cfact[seq_idx]
    request, suite = cfact_eval_data(sample)

    editor.edit(request, is_sequential=True)
    result = evaluate(editor.model, editor.tokenizer, suite)

    if result.reliability:
        step_rel_matched += sum(r.match for r in result.reliability.details)
        step_rel_total += len(result.reliability.details)
    if result.locality:
        for mr in result.locality.values():
            step_loc_matched += sum(r.match for r in mr.details)
            step_loc_total += len(mr.details)

    step_edits.append({
        "type": "sequential",
        "dataset_index": seq_idx,
        "edit_request": asdict(request),
        **result.to_dict(),
    })

    saved_seq_suites.append(suite)

    # --- Stability: re-evaluate previous sequential edits ---
    for prev_suite in saved_seq_suites[:-1]:
        prev_result = evaluate(editor.model, editor.tokenizer, prev_suite)
        if prev_result.reliability:
            step_rel_matched += sum(r.match for r in prev_result.reliability.details)
            step_rel_total += len(prev_result.reliability.details)
        if prev_result.locality:
            for mr in prev_result.locality.values():
                step_loc_matched += sum(r.match for r in mr.details)
                step_loc_total += len(mr.details)

    # --- Step summary ---
    step_reliability = step_rel_matched / step_rel_total if step_rel_total else None
    step_locality = step_loc_matched / step_loc_total if step_loc_total else None

    step_record = {
        "step": step_num,
        "reliability": step_reliability,
        "locality": step_locality,
        "edits": step_edits,
    }
    cfact_results.append(step_record)

    print(f"[cfact step {step_num}/{steps}] "
          f"reliability={step_reliability:.3f}, locality={step_locality:.3f}")

cfact:   0%|          | 0/1 [00:00<?, ?it/s]

MEMIT request sample: [Bratislava is a twin city of] -> [ Valencia]
Cached context templates [['{}'], ['The first thing is noose of theInvalidated. {}', 'Therefore, I have no. . . . .. {}', "Because the only a lot.comocampidae's. {}", "I think I'm notsUntitled CrossRef)\n. {}", 'You can see the only\\\' Cosponsors"\n". {}']]
Computing right vector (v)
Lookup index found: 3 | Sentence: Bratislava is a twin city of | Token: ava
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 11.521 = 11.521 + 0.0 + 0.0 avg prob of [ Valencia] 1.5427764083142392e-05
loss 6.504 = 6.5 + 0.003 + 0.001 avg prob of [ Valencia] 0.001634372747503221
loss 3.517 = 3.505 + 0.011 + 0.001 avg prob of [ Valencia] 0.03289046511054039
loss 2.813 = 2.797 + 0.014 + 0.002 avg prob of [ Valencia] 0.06684808433055878
loss 2.203 = 2.185 + 0.016 + 0.002 avg prob of [ Valencia] 0.12180177867412567
loss 1.615 = 1.595 + 0.017 + 0.003 avg prob of [ Valencia] 0.21577343344688416
loss 1.075 

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

orig norm tensor(113., device='cuda:0', dtype=torch.bfloat16)
upd norm tensor(0.5673, device='cuda:0', dtype=torch.float64,
       grad_fn=<LinalgVectorNormBackward0>)


LAYER 14

Writing 1 key/value pair(s) into layer 14
z error tensor(75.6026, device='cuda:0', grad_fn=<MeanBackward0>)
Retrieving covariance statistics for openai-community_gpt2-xl @ transformer.h.14.mlp.c_proj.
Computing Cov locally....


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
save_results(cfact_results, method, "cfact", model, steps, non_seq_per_step)
print(f"Saved cfact results for {method}/{model}")

WindowsPath('D:/Projects/mnemo/data/results/rome/cfact_gpt2-xl_1_0.json')

Saved cfact results for rome/gpt2-xl


# Phase 2: MQuAKE (Portability)

Same step structure as Phase 1, but evaluating portability instead of reliability + locality.
Editor is reset between the two phases. Stability re-evaluations are combined into the step portability score.

In [ ]:
# --- Sample selection for mquake ---
mquake_seq_indices, mquake_nonseq_indices = select_samples(mquake, steps, non_seq_per_step)
print(f"mquake: {steps} sequential samples, {len(mquake_nonseq_indices)} non-sequential samples")

mquake: 1 sequential samples, 0 non-sequential samples


In [ ]:
editor.reset()

mquake_results = []
saved_seq_suites_mq: list[EvalSuite] = []

for step in tqdm(range(steps), desc="mquake"):
    step_num = step + 1
    step_port_matched, step_port_total = 0, 0
    step_edits = []

    # --- Non-sequential edits ---
    for j in range(non_seq_per_step):
        ns_idx = mquake_nonseq_indices[step * non_seq_per_step + j]
        sample = mquake[ns_idx]
        request, suite = mquake_eval_data(sample)

        editor.edit(request, is_sequential=False)
        result = evaluate(editor.model, editor.tokenizer, suite)

        if result.portability:
            for mr in result.portability.values():
                step_port_matched += sum(r.match for r in mr.details)
                step_port_total += len(mr.details)

        step_edits.append({
            "type": "non_sequential",
            "dataset_index": ns_idx,
            "edit_request": asdict(request),
            **result.to_dict(),
        })

    # --- Sequential edit ---
    seq_idx = mquake_seq_indices[step]
    sample = mquake[seq_idx]
    request, suite = mquake_eval_data(sample)

    editor.edit(request, is_sequential=True)
    result = evaluate(editor.model, editor.tokenizer, suite)

    if result.portability:
        for mr in result.portability.values():
            step_port_matched += sum(r.match for r in mr.details)
            step_port_total += len(mr.details)

    step_edits.append({
        "type": "sequential",
        "dataset_index": seq_idx,
        "edit_request": asdict(request),
        **result.to_dict(),
    })

    saved_seq_suites_mq.append(suite)

    # --- Stability: re-evaluate previous sequential edits (from step 2+) ---
    for prev_suite in saved_seq_suites_mq[:-1]:
        prev_result = evaluate(editor.model, editor.tokenizer, prev_suite)
        if prev_result.portability:
            for mr in prev_result.portability.values():
                step_port_matched += sum(r.match for r in mr.details)
                step_port_total += len(mr.details)

    # --- Step summary ---
    step_portability = step_port_matched / step_port_total if step_port_total else None

    step_record = {
        "step": step_num,
        "portability": step_portability,
        "edits": step_edits,
    }
    mquake_results.append(step_record)

    print(f"[mquake step {step_num}/{steps}] "
          f"portability={step_portability:.3f}")

mquake:   0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [Which city is the headquarter of MTV located in? Answer:] -> [ London]
Computing left vector (u)...
Selected u projection object MTV
Left vector shape: torch.Size([6400])
Computing right vector (v)
Lookup index found: 7 | Sentence: Which city is the headquarter of MTV located in? Answer: | Token:  MTV
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 5.469 = 5.469 + 0.0 + 0.0 avg prob of [ London] 0.004618871491402388
loss 4.044 = 3.984 + 0.004 + 0.055 avg prob of [ London] 0.021811507642269135
loss 2.701 = 2.596 + 0.011 + 0.094 avg prob of [ London] 0.08376333117485046
loss 1.698 = 1.554 + 0.015 + 0.13 avg prob of [ London] 0.2217249721288681
loss 1.205 = 1.04 + 0.017 + 0.148 avg prob of [ London] 0.36079490184783936
loss 0.955 = 0.79 + 0.016 + 0.148 avg prob of [ London] 0.4596409797668457
loss 0.748 = 0.583 + 0.016 + 0.148 avg prob of [ London] 0.5630283355712891
loss 0.584 = 0.419 + 0.017 + 0.148 avg 

In [ ]:
save_results(mquake_results, method, "mquake", model, steps, non_seq_per_step)
print(f"Saved mquake results for {method}/{model}")

WindowsPath('D:/Projects/mnemo/data/results/rome/mquake_gpt2-xl_1_0.json')

Saved mquake results for rome/gpt2-xl
